<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5: Eager and Lazy APIs

This notebook accompanies Chapter 5 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

Polars gives you two ways to work with data, and choosing between them is one of the most important decisions you make:

- the **eager API** runs every step immediately on a `DataFrame`
- the **lazy API** records your steps as a *plan* on a `LazyFrame`, optimizes the plan, and runs it only when you ask for the result

We use one real question throughout, answered both ways on New York City **yellow taxi** trip data:

> *Which three taxi vendors earn the most revenue per kilometer (or mile) traveled?*

By the end of this notebook you will be able to:

- explain what each API is best at, and pick one for a given task
- read a **query plan** and see which optimizations Polars applied
- **measure** the performance difference instead of taking it on faith
- see how the lazy API catches type errors **before** doing the work
- avoid recomputing an expensive result, using caching and `collect_all`
- convert between DataFrames and LazyFrames, and join them
- know which methods exist on only one of the two
- process data **bigger than memory** with streaming and `sink_*`

**How to use this notebook:** run the cells from top to bottom. Some cells *intentionally* trigger an error to teach a point; those errors are caught and printed, so they won't stop the notebook.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior. The `graph` extra lets Polars draw query plans as diagrams. We use `%pip` so the package is installed into the running notebook kernel.

In [ ]:
%pip install -q "polars[graph]==2.1.*"

### Imports and a small helper

The helper function `timed` runs a function several times and reports the **best** time. A single timing is unreliable: the very first read of a file is slower because the data isn't in the operating system's cache yet. Repeating and taking the best of a few runs gives a fairer comparison.

In [ ]:
from pathlib import Path
from time import perf_counter
import urllib.request

import polars as pl
from polars.testing import assert_frame_equal


print(f"Polars {pl.__version__}")


timings: list[dict] = []


def timed(label: str, func, repeat: int = 3):
    """Run func() `repeat` times, record and print the best time, and return the last result."""
    durations = []
    for _ in range(repeat):
        start = perf_counter()
        result = func()
        durations.append(perf_counter() - start)
    timings.append({"approach": label, "best_seconds": round(min(durations), 3)})
    print(f"{label}: best of {repeat} = {min(durations):.2f} s")
    return result


### Download the taxi data

The data is one Parquet file per month of 2022. Each month has roughly three million trips, so the *full year* has about 40 million rows and 19 columns. Loading all of that eagerly can need **many gigabytes of memory**, which can crash a free Colab session (about 12 GB).

In this notebook we are downloading only the **first three months**. Change `MONTHS` to `range(1, 13)` for the whole year if you want to try working with the whole year's worth of data.

Files that are already downloaded are skipped. Each file is written to a temporary name and renamed only when complete, so an interrupted download can't leave a corrupt file behind. An HTTP error raises an exception instead of being saved as if it were data.

In [ ]:
MONTHS = range(1, 4)  # January to March. Use range(1, 13) for the full year.

TAXI_DIR = Path("sample_data/taxi")
BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/taxi"


def download_taxi_files(months) -> list[str]:
    """Download the monthly Parquet files (if needed) and return their paths."""
    TAXI_DIR.mkdir(parents=True, exist_ok=True)
    paths = []
    for month in months:
        name = f"yellow_tripdata_2022-{month:02d}.parquet"
        path = TAXI_DIR / name
        if not path.exists():
            temporary = path.with_suffix(".tmp")
            urllib.request.urlretrieve(f"{BASE_URL}/{name}", temporary)
            temporary.rename(path)
        print(f"{name}: {path.stat().st_size / 1e6:.1f} MB")
        paths.append(str(path))
    return paths


TAXI_FILES = download_taxi_files(MONTHS)

## Eager API: DataFrame

In the **eager** API every line runs immediately on data that is already in memory. That makes it ideal for **exploration**: after each step you can look at the intermediate result and decide what to do next. This is the style that pandas users know.

First, read the files into one DataFrame. `read_parquet` loads **every column** of **every file** into memory. Check how big that is before doing anything else:

In [ ]:
trips_df = pl.read_parquet(TAXI_FILES)

print(f"shape: {trips_df.shape}")
print(f"memory: {trips_df.estimated_size('gb'):.2f} GB")
print(f"{len(trips_df.columns)} columns: {trips_df.columns}")

For our question we only need three of those 19 columns: `VendorID`, `total_amount` (what the passenger paid), and `trip_distance`.

**Step 1: totals per vendor.** `group_by` splits the trips by vendor, and `agg` computes one summary per group. We name exactly the two columns we want to sum.

(The book calls `group_by("VendorID").sum()`, which sums *every* column. Naming the columns is clearer, and in lazy mode it lets Polars skip reading the rest.)

In [ ]:
sum_per_vendor_df = (
    trips_df.group_by("VendorID")
            .agg(
                pl.col("total_amount").sum(),
                pl.col("trip_distance").sum(),
            )
)

sum_per_vendor_df.sort("VendorID")

**Step 2: revenue per distance.** Divide the two totals. This is *total* revenue over *total* distance, which differs from averaging each trip's own ratio. Totals weight long trips more, which is usually what you want for "how much does a vendor earn per unit traveled".

In [ ]:
income_per_distance_df = sum_per_vendor_df.select(
    "VendorID",
    income_per_distance=pl.col("total_amount") / pl.col("trip_distance"),
)

income_per_distance_df.sort("VendorID")

**Step 3: the top three.** Sort in descending order and keep the first three rows.

In [ ]:
top_three_df = (
    income_per_distance_df.sort(
        "income_per_distance",
        descending=True
    ).head(3)
)

top_three_df

Working this way, you saw the data at every step. When you are exploring and don't yet know what you're looking for, that is the great strength of the eager API.

It has a cost, though: Polars runs each step on its own and can't look ahead. It loaded all 19 columns although the question needed three.


## Lazy API: LazyFrame

The **lazy** API works differently. You build a **query**, a recipe of steps, and nothing runs until you call `collect()`. Because Polars sees the whole recipe before running it, it can *optimize*:

- read only the **columns** the query needs
- read only the **rows** that pass your filters, as early as possible (at the scan)
- skip work whose result is never used
- reorder and combine steps

`scan_parquet` is the lazy counterpart of `read_parquet`. It reads only the file's metadata (such as the schema) and returns a `LazyFrame`. We name lazy variables with an `_lf` suffix and eager ones with `_df`, so you can always tell which is which.

In [ ]:
trips_lf = pl.scan_parquet(TAXI_FILES)

trips_lf

A LazyFrame holds no data, but it **knows its schema**. `collect_schema()` returns the column types by reading only metadata. (Prefer it over the `schema` or `columns` attributes on a LazyFrame: it makes explicit that the schema may have to be resolved.)

In [ ]:
schema = trips_lf.collect_schema()
print(schema)


In [ ]:
for name, dtype in list(schema.items()):
    print(f"  {name}: {dtype}")

Now the same three steps. The code is nearly identical to the eager version. Only the starting point changed, and `collect()` at the end triggers the work. Nothing is read from disk until then.

In [ ]:
sum_per_vendor_lf = (
    trips_lf.group_by("VendorID")
            .agg(
                pl.col("total_amount").sum(),
                pl.col("trip_distance").sum(),
            )
)

income_per_distance_lf = sum_per_vendor_lf.select(
    "VendorID",
    income_per_distance=pl.col("total_amount") / pl.col("trip_distance"),
)

top_three_lf = (
    income_per_distance_lf.sort(
        "income_per_distance",
        descending=True
    ).head(3)
)

top_three_lf

### Reading the query plan

`explain()` shows the **optimized plan**. Read it **from the bottom up**: the bottom is the first step (the scan), and the top is the last. Look at the scan line at the bottom: Polars should report that it reads only **3 of the 19 columns** (a "projection"). The columns we never mention are never loaded.

Compare with `optimized=False`, the plan exactly as we wrote it.

In [ ]:
print(top_three_lf.explain())

In [ ]:
print(top_three_lf.explain(optimized=False))

The plan can also be drawn as a diagram. It needs Graphviz and matplotlib, so we guard the call. If it fails, the text plans above contain the same information.

In [ ]:
top_three_lf.show_graph()

Finally, `collect()` runs the plan. The answer should match the eager result exactly. We check that rather than assume it:

In [ ]:
top_three_collected_df = top_three_lf.collect()

In [ ]:
# would throw an error if False
assert_frame_equal(top_three_df, top_three_collected_df)

In [ ]:
top_three_collected_df

### Lazy queries fail fast

Because a LazyFrame knows its schema, Polars can check your query *before* running it. Here we make a deliberate mistake: calling a string method (`str.slice`) on a column that holds numbers.

First on a tiny in-memory LazyFrame, where we expect a schema error:

In [ ]:
names_lf = pl.LazyFrame({"name": ["Alice", "Bob", "Charlie"], "age": [25, 30, 35]})

names_lf.with_columns(sliced_age=pl.col("age").str.slice(1, 3)).collect()

With eager execution you'd typically discover this only after earlier steps have already run. For a long-running pipeline, catching mistakes in milliseconds instead of hours is a big productivity win.

## Performance Differences

Is lazy really faster? Let's measure, carefully.

To make the comparison fair, we write the query **once** as a function. A neat property of Polars is that DataFrames and LazyFrames share most of their methods, so the *same function* works on either, returning the same kind of object it was given.

In [ ]:
def top_three_vendors(trips):
    """Top three vendors by total revenue per total distance. Works on a DataFrame or a LazyFrame."""
    return (
        trips.group_by("VendorID")
        .agg(
            pl.col("total_amount").sum(),
            pl.col("trip_distance").sum(),
        )
        .select(
            "VendorID",
            income_per_distance=pl.col("total_amount") / pl.col("trip_distance"),
        )
        .sort("income_per_distance", descending=True)
        .head(3)
    )

We time three ways of running it. Each includes **reading the data**, because avoiding unnecessary reading is exactly where lazy evaluation helps:

1. **Eager, all columns:** `read_parquet` then query (what the first half of this notebook did)
2. **Eager, hand-picked columns:** pass `columns=[...]` to `read_parquet`. You do the optimization *manually*
3. **Lazy:** `scan_parquet`, query, `collect()`. Polars does it for you

Results depend on your machine and on caching, so treat the numbers as indicative.

In [ ]:
timings.clear()
NEEDED_COLUMNS = ["VendorID", "total_amount", "trip_distance"]

result_eager_df = timed(
    "eager, all columns",
    lambda: top_three_vendors(pl.read_parquet(TAXI_FILES)),
)
result_pruned_df = timed(
    "eager, hand-picked columns",
    lambda: top_three_vendors(pl.read_parquet(TAXI_FILES, columns=NEEDED_COLUMNS)),
)
result_lazy_df = timed(
    "lazy",
    lambda: top_three_vendors(pl.scan_parquet(TAXI_FILES)).collect(),
)

assert_frame_equal(result_eager_df, result_pruned_df)
assert_frame_equal(result_eager_df, result_lazy_df)
print("All three approaches give the same answer.")

In [ ]:
pl.DataFrame(timings).with_columns(
    relative_to_eager=(pl.col("best_seconds") / pl.col("best_seconds").first()).round(2)
)

What to take from this:

- The lazy version should be much faster than the naive eager one, because it only reads three columns.
- The eager version with `columns=[...]` can get close, **if you remember to do it**. Lazy makes that optimization automatic, and it extends to filters, slices, and joins that are tedious to optimize by hand.
- For small data, or when you need to look at intermediate results, the difference doesn't matter. Pick the API that helps you think.


### A more compact query

The two-stage version (aggregate, then divide) can be written as one aggregation. Using expressions inside `agg` lets you compute the ratio of the sums directly:

In [ ]:
compact_lf = (
    trips_lf.group_by("VendorID")
    .agg(income_per_distance=pl.col("total_amount").sum() / pl.col("trip_distance").sum())
    .sort("income_per_distance", descending=True)
    .head(3)
)

assert_frame_equal(compact_lf.collect(), result_lazy_df)

In [ ]:
compact_lf.collect()

For "the top *k* rows" there is also `top_k`, which is typically faster than sorting everything. It returns the rows in **no guaranteed order**, so sort afterwards if order matters:

In [ ]:
(
    trips_lf.group_by("VendorID")
    .agg(income_per_distance=pl.col("total_amount").sum() / pl.col("trip_distance").sum())
    .top_k(3, by="income_per_distance")
    .sort("income_per_distance", descending=True)
    .collect()
)

## Calling `collect()` repeatedly recomputes everything

A LazyFrame is a *recipe*, not a result. Every time you call `collect()`, Polars cooks the whole recipe again. That is a nasty surprise if the recipe contains an expensive step.

We can make this visible. The function below is a stand-in for "something expensive". It prints a message each time it runs, so you can count how often the work is repeated.

In [ ]:
def expensive(series: pl.Series) -> pl.Series:
    print("  expensive computation running...")
    return series * 2


lf = pl.LazyFrame({"col1": [1, 2, 3], "col2": [4, 5, 6]}).with_columns(
    col1=pl.col("col1").map_batches(expensive, return_dtype=pl.Int64)
)

In [ ]:
print("First collect:")
lf.collect()
print("Second collect:")
lf.collect()

The message printed **twice**: the work was done again from scratch.


### Caching a result

If you will use a result several times, **materialize it once**: call `collect()`, then `.lazy()` to get a LazyFrame that points at the stored data. Later queries start from the stored result and skip the expensive step. (Name the cached frame differently from the original so it's clear which is which.)

In [ ]:
# caching the collected results
lf_cached = lf.collect().lazy()

In [ ]:
lf_cached.collect()

In [ ]:
lf_cached.with_columns(col1=pl.col("col1") + 1).collect()

The expensive message appeared only once, during caching, and not for the two follow-up queries.


### `collect_all`: run several queries together

Often you have *several* queries that start from the same expensive result. `pl.collect_all([...])` executes them together, so Polars has the chance to compute the shared part once. Check how many times the message prints in each version. (How much sharing happens depends on the optimizer, so check the documentation for its current behavior.)

In [ ]:
total_lf = lf.select(total=pl.col("col1").sum())
mean_lf = lf.select(average=pl.col("col1").mean())


In [ ]:
# collected separately
total_df = total_lf.collect()
mean_df = mean_lf.collect()

In [ ]:
# collected together
total_df, mean_df = pl.collect_all([total_lf, mean_lf])
total_df, mean_df

**Rule of thumb:** use plain `collect()` once at the end of a pipeline. Use `collect().lazy()` when you need an intermediate result *more than once*. Use `collect_all` when you have several final queries over shared work. And don't call `collect()` in the middle of a pipeline "just in case": that gives up the optimizations the lazy API exists for (Chapter 18 lists this as an antipattern).

LazyFrames also have a `cache()` method. It marks a point *inside a single query plan* whose result should be stored for other branches of the same query. It is a different tool from `collect().lazy()`, which stores a result *between* queries.

## Tips and Tricks

### Going from LazyFrame to DataFrame and back

Switching between the two APIs takes one method call:

- `df.lazy()` turns a DataFrame into a LazyFrame. No computation happens. The data in memory simply becomes the starting point of a new plan.
- `lf.collect()` turns a LazyFrame into a DataFrame by executing the plan.

In [ ]:
small_df = pl.DataFrame({"a": [1, 2, 3], "b": [4.0, 5.0, 6.0]})
small_lf = small_df.lazy()
collected_df = small_lf.select(c=pl.col("a") + pl.col("b")).collect()

print(f"{type(small_df).__name__} -> .lazy() -> {type(small_lf).__name__} -> .collect() -> {type(collected_df).__name__}")

A good working style is to **explore eagerly** on a sample, then **write the final pipeline lazily** and collect once at the end.

### Joining a DataFrame with a LazyFrame

A join needs both sides to be the same kind. Joining a LazyFrame with a DataFrame isn't allowed. A typical case: a big LazyFrame and a small lookup table you built as a DataFrame.

In [ ]:
big_sales_lf = pl.LazyFrame(
    {
        "sale_id": [101, 102, 103],
        "amount": [250, 150, 300]
    }
)
sales_metadata_df = pl.DataFrame(
    {
        "sale_id": [101, 102, 103],
        "category": ["A", "B", "A"]
    }
)

big_sales_lf.join(sales_metadata_df, on="sale_id").collect()

The fix: make the DataFrame lazy with `.lazy()` (we recommend this, so the whole query stays lazy and optimizable) or collect the LazyFrame. We also add `validate="1:1"`, which declares that each `sale_id` should appear exactly once on each side. Polars raises an error if the data violates that, instead of silently multiplying rows (Chapter 14 covers join validation).

In [ ]:
big_sales_lf.join(sales_metadata_df.lazy(), on="sale_id", validate="1:1").collect()

### Not every method exists on both

A DataFrame holds data; a LazyFrame doesn't. So anything that needs to *know the data* right now (its row count, a pivot's new column names) is awkward or impossible lazily.

## Processing data larger than memory

The lazy API can also run in **streaming** mode, which processes the data in **chunks** instead of loading it all at once. That moves the limit on what you can process from the size of your RAM to the size of your disk. There are two ways to use it:

1. `collect(engine="streaming")` runs the query in chunks but still returns the final result in memory, so **the result itself must fit in RAM**. (Older versions used `collect(streaming=True)`; the `engine` argument is the current way.)
2. `sink_parquet` (and `sink_csv`, `sink_ipc`, `sink_ndjson`) runs the query in chunks and **writes the result straight to disk**, so even a huge *result* works.

This mode was marked experimental in the book. Check the documentation for its status in your version. Let's try both on the taxi data.

First, the streaming engine. The answer must match the default engine's:

In [ ]:
streamed_df = top_three_vendors(trips_lf).collect(engine="streaming")
assert_frame_equal(streamed_df, result_lazy_df)


In [ ]:
streamed_df

Now `sink_parquet`. We keep only long trips and write them to disk without ever holding the result in memory. Then we read the file *lazily* to count the rows, which again avoids loading it.

In [ ]:
OUTPUT_DIR = Path("sample_data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
long_trips_path = OUTPUT_DIR / "long_trips.parquet"

(
    trips_lf.filter(pl.col("trip_distance") > 10)
            .select("VendorID", "trip_distance", "total_amount")
            .sink_parquet(long_trips_path)
)

n_rows = pl.scan_parquet(long_trips_path).select(pl.len()).collect().item()
print(f"Wrote {n_rows:,} long trips to {long_trips_path} ({long_trips_path.stat().st_size / 1e6:.1f} MB)")


In [ ]:
! ls -lh sample_data/output/long_trips.parquet

## Which API should I use?

| Situation | Use | Why |
|---|---|---|
| Exploring a new dataset, checking intermediate results | **Eager** | Immediate feedback after every step |
| Small data where speed doesn't matter | **Eager** | Simpler, and optimization has nothing to gain |
| Production pipeline or scheduled job | **Lazy** | Optimization, and mistakes caught before the work starts |
| Large files, only some columns or rows needed | **Lazy** | Reads only what the query needs |
| Result is bigger than RAM | **Lazy + `sink_*`** | Streams chunks to disk |
| Need an operation only DataFrames have | **Collect, then eager** | Do the lazy part first, collect, then finish eagerly |

A practical workflow: **prototype eagerly** on a sample, then **rewrite as a lazy query** that starts with `scan_*` and ends with a single `collect()` (or `sink_*`).

## Takeaways

- The **eager** API (`DataFrame`) runs each step immediately: ideal for exploration, since you can inspect every intermediate result.
- The **lazy** API (`LazyFrame`) records a plan, optimizes it, and runs it on `collect()`: ideal for performance and for large data.
- Read the **query plan** with `explain()` from the bottom up, and look for the columns and filters pushed into the scan.
- A LazyFrame knows its schema, so many mistakes are caught **before** any data is processed.
- **Measure** performance (repeat runs, include reading the data) and **verify** that variants agree with `assert_frame_equal`.
- Every `collect()` recomputes the plan. Cache with `collect().lazy()`, share work with `collect_all`, and otherwise collect once at the end.
- Switch with `df.lazy()` and `lf.collect()`. Both sides of a join must be the same kind. Prefer `.lazy()`.
- Some methods exist on only one type, so check, and prefer expression functions (such as `pl.sum_horizontal`) that work in both.
- **Streaming** and `sink_*` let you process more data than fits in memory.

### Next up

In Chapter 6 we look at reading and writing data in many formats, including CSV, Excel, JSON, and databases.

### Try it yourself

1. Set `MONTHS = range(1, 7)` and re-run the timing comparison. How does the gap between the three approaches change?


2. Add a filter for trips longer than 1 km/mile *before* aggregating, once in the eager version and once in the lazy version. In the lazy plan, where does the filter end up? (Look for it in the scan.)


3. Write a lazy query that reports the average `tip_amount` per `payment_type`, and check with `explain()` how many columns it reads.


4. Replace the the code in the join section with a version that calls `collect()` on the LazyFrame instead of calling `.lazy()` on the DataFrame. Which is better when the LazyFrame is huge, and why?
